<a href="https://colab.research.google.com/github/avindumihisara0229-code/ErgoSense/blob/Avindu/Posture/TrainPosture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mediapipe
!pip install mediapipe protobuf numpy
!pip install scikit-learn pandas opencv-python tqdm -q
!pip uninstall -y mediapipe protobuf numpy

!pip install mediapipe protobuf numpy
!pip install scikit-learn pandas opencv-python tqdm -q
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
from google.colab import drive
from tqdm.notebook import tqdm

drive.mount('/content/drive')
print("✅ Step 1: Setup complete. Libraries installed and Drive mounted.")

Define Paths

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/'

# Define the subfolders for correct and incorrect postures
correct_folders = [
    os.path.join(DATASET_PATH, 'old', '0'),
    os.path.join(DATASET_PATH, 'old', '2'),
    os.path.join(DATASET_PATH, 'old', '3')
]

incorrect_folders = [
    os.path.join(DATASET_PATH, 'old', 'bad', '0'),
    os.path.join(DATASET_PATH, 'old', 'bad', '2'),
    os.path.join(DATASET_PATH, 'old', 'bad', '3')
]

print("✅ Step 2: File paths are set.")

Feature Extraction

In [ ]:
mp_pose = mp.solutions.pose
# We must define the pose_model *before* the function that uses it.
pose_model = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

def extract_posture_landmarks_from_path(pose_estimator, image_path):
    image = cv2.imread(image_path)
    if image is None: return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose_estimator.process(image_rgb)

    if not results.pose_landmarks: return None

    landmarks = results.pose_landmarks.landmark
    try:
        left_shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
        left_ear = [landmarks[mp_pose.PoseLandmark.LEFT_EAR.value].x, landmarks[mp_pose.PoseLandmark.LEFT_EAR.value].y]
        left_hip = [landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].x, landmarks[mp_pose.PoseLandmark.LEFT_HIP.value].y]
        left_knee = [landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].x, landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value].y]

        neck_angle = calculate_angle(left_ear, left_shoulder, left_hip)
        back_angle = calculate_angle(left_shoulder, left_hip, left_knee)

        return [neck_angle, back_angle]
    except:
        return None

print("✅ Step 3: Feature extraction logic defined.")